<a href="https://colab.research.google.com/github/HudaSaffo/fashion-recommendation-system/blob/week3-segmentation/image_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Segmentation

This notebook creates transparent product cutouts from the image cache. The cutouts will be used later for collage composition and image-based search.

## 1. Install and Verify Libraries

Install the required segmentation and image-processing libraries, then confirm that Pillow is working correctly.

In [1]:
%pip install -q --no-cache-dir --force-reinstall "Pillow==12.3.0"
%pip install -q git+https://github.com/facebookresearch/segment-anything.git
%pip install -q opencv-python-headless pandas tqdm matplotlib rembg onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 152.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 140.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 43.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.67.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you 

In [1]:
import PIL
from PIL import Image, ImageText

print(PIL.__version__)
print('Pillow is working.')

12.3.0
Pillow is working.


## 2. Set Up Google Drive, Data, and Output Folders

Load the product-image cache and the cleaned catalog. Define the folders used to save masks, transparent cutouts, review files, and batch results.

In [2]:
import html
import shutil
import urllib.request
from urllib.error import HTTPError
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

INPUT_DIR = Path('/content/drive/MyDrive/polyvore_image_cache')
CATALOG_PATH = Path('/content/drive/MyDrive/catalog.parquet')
OUTPUT_DIR = Path('/content/drive/MyDrive/polyvore_segmented_cache')

MASK_DIR = OUTPUT_DIR / 'masks'
CUTOUT_DIR = OUTPUT_DIR / 'cutouts'
REVIEW_DIR = OUTPUT_DIR / 'review'

for directory in (MASK_DIR, CUTOUT_DIR, REVIEW_DIR):
    directory.mkdir(parents=True, exist_ok=True)

catalog = pd.read_parquet(CATALOG_PATH)
catalog['item_id'] = catalog['item_id'].astype(str)
CATALOG_BY_ITEM_ID = catalog.set_index('item_id').to_dict('index')

MAX_IMAGES = None
CUTOUT_PADDING_PX = 12
PIPELINE_VERSION = 'week3_v4'

CONFIDENT_NONWHITE_COLORS = {
    'black', 'blue', 'red', 'green', 'yellow', 'pink', 'gold', 'silver',
    'grey', 'gray', 'brown', 'beige',
}

MIN_MASK_AREA_RATIO = 0.008
MAX_MASK_AREA_RATIO = 0.92
MIN_BBOX_AREA_RATIO = 0.035
MIN_BBOX_SIDE_RATIO = 0.07
MIN_PROXY_COVERAGE = 0.42

print(f'Catalog records: {len(CATALOG_BY_ITEM_ID):,}')
print(f'Image cache:     {INPUT_DIR}')
print(f'Output folder:   {OUTPUT_DIR}')
print('Corrected smoke test is ON.')

Catalog records: 3,762
Image cache:     /content/drive/MyDrive/polyvore_image_cache
Output folder:   /content/drive/MyDrive/polyvore_segmented_cache
Corrected smoke test is ON.


## 3. Download and Initialize Segmentation Models

Download the official SAM ViT-B checkpoint if it is not already available. Then initialize SAM and rembg U2-Net to generate product masks.

In [3]:
CHECKPOINT = Path('/content/sam_vit_b_01ec64.pth')
CHECKPOINT_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'

def download_sam_checkpoint(url, destination):
    partial = destination.with_suffix(destination.suffix + '.part')
    request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        with urllib.request.urlopen(request, timeout=120) as response, open(partial, 'wb') as output:
            shutil.copyfileobj(response, output)
        partial.replace(destination)
    except HTTPError as error:
        partial.unlink(missing_ok=True)
        raise RuntimeError(f'SAM checkpoint download was denied ({error.code}).') from error

if not CHECKPOINT.exists() or CHECKPOINT.stat().st_size < 100_000_000:
    CHECKPOINT.unlink(missing_ok=True)
    print('Downloading official SAM ViT-B checkpoint...')
    download_sam_checkpoint(CHECKPOINT_URL, CHECKPOINT)

import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
from rembg import new_session, remove

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
sam = sam_model_registry['vit_b'](checkpoint=str(CHECKPOINT)).to(device=DEVICE)
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,
    pred_iou_thresh=0.80,
    stability_score_thresh=0.88,
    crop_n_layers=1,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=100,
)
rembg_session = new_session('u2net')
print(f'SAM + rembg ready (SAM device: {DEVICE}).')

  0%|                                               | 0.00/176M [00:00<?, ?B/s]

SAM + rembg ready (SAM device: cuda).


## 4. Define Segmentation, Mask Selection, and Quality Checks

Create the helper functions used to identify the main product, generate mask candidates, select the best full-product mask, and reject unreliable cutouts.

In [4]:
def white_background_mask(rgb, threshold=245):
    return np.all(rgb >= threshold, axis=2)

def has_clean_white_background(rgb):
    white = white_background_mask(rgb)
    height, width = white.shape
    border_width = max(2, min(height, width) // 30)
    border = np.concatenate([
        white[:border_width].ravel(),
        white[-border_width:].ravel(),
        white[:, :border_width].ravel(),
        white[:, -border_width:].ravel(),
    ])
    return bool(
        border.mean() >= 0.97
        and white.mean() >= 0.55
    )

def color_route(item_metadata):
    color = str(item_metadata.get('color', '')).strip().lower()
    if color in CONFIDENT_NONWHITE_COLORS:
        return 'shortcut_eligible_nonwhite'
    return 'hybrid_required'

def should_use_white_background_shortcut(rgb, item_metadata):
    return (
        color_route(item_metadata) == 'shortcut_eligible_nonwhite'
        and has_clean_white_background(rgb)
    )

def clean_mask(mask):
    binary = (np.asarray(mask) > 0).astype(np.uint8)
    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        np.ones((3, 3), np.uint8),
        iterations=1
    )
    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary, 8)
    if count <= 1:
        return binary.astype(bool)
    total = int(binary.sum())
    min_component = max(20, int(total * 0.003))
    kept = np.zeros_like(binary)
    for label in range(1, count):
        if stats[label, cv2.CC_STAT_AREA] >= min_component:
            kept[labels == label] = 1
    return kept.astype(bool)

def foreground_from_white(rgb):
    return clean_mask(~white_background_mask(rgb))

def rembg_product_mask(rgb):
    mask_image = remove(
        Image.fromarray(rgb),
        session=rembg_session,
        only_mask=True,
        alpha_matting=True,
        alpha_matting_foreground_threshold=240,
        alpha_matting_background_threshold=10,
        alpha_matting_erode_size=8,
    )
    alpha = np.asarray(mask_image.convert('L'))
    return clean_mask(alpha >= 48)

def border_foreground_proxy(rgb):
    height, width = rgb.shape[:2]
    border_width = max(2, min(height, width) // 25)
    border_pixels = np.concatenate([
        rgb[:border_width].reshape(-1, 3),
        rgb[-border_width:].reshape(-1, 3),
        rgb[:, :border_width].reshape(-1, 3),
        rgb[:, -border_width:].reshape(-1, 3),
    ], axis=0)
    background = np.median(
        border_pixels.astype(np.float32),
        axis=0
    )
    distance = np.linalg.norm(
        rgb.astype(np.float32) - background,
        axis=2
    )
    threshold = max(
        10.0,
        float(np.percentile(distance, 65)) * 0.35
    )
    proxy = distance > threshold
    proxy = cv2.morphologyEx(
        proxy.astype(np.uint8),
        cv2.MORPH_CLOSE,
        np.ones((5, 5), np.uint8),
        iterations=2
    )
    return clean_mask(proxy)

def bbox_from_mask(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return (
        int(xs.min()),
        int(ys.min()),
        int(xs.max()) + 1,
        int(ys.max()) + 1,
    )

def mask_metrics(mask, proxy=None):
    height, width = mask.shape
    bbox = bbox_from_mask(mask)
    if bbox is None:
        return {
            'valid': False,
            'reasons': ['empty_mask'],
        }
    x1, y1, x2, y2 = bbox
    area_ratio = float(mask.mean())
    bbox_width_ratio = (x2 - x1) / width
    bbox_height_ratio = (y2 - y1) / height
    bbox_area_ratio = ((x2 - x1) * (y2 - y1)) / (height * width)
    proxy_coverage = None
    if proxy is not None and proxy.sum() >= height * width * 0.002:
        proxy_coverage = float(
            np.logical_and(mask, proxy).sum()
            / max(1, proxy.sum())
        )
    reasons = []
    if not MIN_MASK_AREA_RATIO <= area_ratio <= MAX_MASK_AREA_RATIO:
        reasons.append('mask_area_out_of_range')
    if bbox_area_ratio < MIN_BBOX_AREA_RATIO:
        reasons.append('bbox_too_small')
    if min(bbox_width_ratio, bbox_height_ratio) < MIN_BBOX_SIDE_RATIO:
        reasons.append('suspicious_thin_fragment')
    if (
        proxy_coverage is not None
        and proxy_coverage < MIN_PROXY_COVERAGE
    ):
        reasons.append('low_whole_object_coverage')
    return {
        'valid': not reasons,
        'reasons': reasons,
        'area_ratio': area_ratio,
        'bbox_width_ratio': bbox_width_ratio,
        'bbox_height_ratio': bbox_height_ratio,
        'bbox_area_ratio': bbox_area_ratio,
        'proxy_coverage': proxy_coverage,
    }

def candidate_score(mask, proxy, predicted_iou=0.0, stability=0.0):
    metrics = mask_metrics(mask, proxy)
    if metrics.get('area_ratio') is None:
        return -np.inf, metrics
    coverage = (
        metrics['proxy_coverage']
        if metrics['proxy_coverage'] is not None
        else 0.5
    )
    side_balance = min(
        metrics['bbox_width_ratio'],
        metrics['bbox_height_ratio']
    )
    score = (
        3.5 * coverage
        + 1.0 * min(metrics['bbox_area_ratio'], 0.75)
        + 0.35 * min(metrics['area_ratio'], 0.65)
        + 0.25 * side_balance
        + 0.25 * predicted_iou
        + 0.15 * stability
    )
    if not metrics['valid']:
        score -= 2.5 + 0.4 * len(metrics['reasons'])
    return score, metrics

def choose_product_mask(rgb, item_metadata):
    proxy = border_foreground_proxy(rgb)
    candidates = []
    if should_use_white_background_shortcut(rgb, item_metadata):
        candidates.append((
            'white_background_shortcut',
            foreground_from_white(rgb),
            1.0,
            1.0,
        ))
    try:
        candidates.append((
            'rembg_u2net',
            rembg_product_mask(rgb),
            0.0,
            0.0,
        ))
    except Exception as exc:
        print(f'rembg warning: {exc}')
    for proposal in mask_generator.generate(rgb):
        candidates.append((
            'sam_vit_b',
            clean_mask(proposal['segmentation']),
            float(proposal.get('predicted_iou', 0)),
            float(proposal.get('stability_score', 0)),
        ))
    ranked = []
    for method, mask, predicted_iou, stability in candidates:
        score, metrics = candidate_score(
            mask,
            proxy,
            predicted_iou,
            stability
        )
        ranked.append((score, method, mask, metrics))
    ranked.sort(key=lambda row: row[0], reverse=True)
    if not ranked:
        raise ValueError('no segmentation candidates')
    score, method, mask, metrics = ranked[0]
    if not metrics['valid']:
        raise ValueError(
            'quality_gate_failed: '
            + ','.join(metrics['reasons'])
        )
    metrics['selection_score'] = float(score)
    return mask, method, metrics

def padded_bbox(bbox, image_width, image_height, padding):
    x1, y1, x2, y2 = bbox
    return (
        max(0, x1 - padding),
        max(0, y1 - padding),
        min(image_width, x2 + padding),
        min(image_height, y2 + padding),
    )

def build_output_record(mask, item_id, source_path, method, item_metadata, quality):
    original_bbox = bbox_from_mask(mask)
    if original_bbox is None:
        return None
    height, width = mask.shape
    crop_bbox = padded_bbox(
        original_bbox,
        width,
        height,
        CUTOUT_PADDING_PX
    )
    x1, y1, x2, y2 = original_bbox
    cx1, cy1, cx2, cy2 = crop_bbox
    return {
        'pipeline_version': PIPELINE_VERSION,
        'item_id': item_id,
        'source_path': str(source_path),
        'mask_path': str(MASK_DIR / f'{item_id}.png'),
        'cutout_path': str(CUTOUT_DIR / f'{item_id}.png'),
        'method': method,
        'color_route': color_route(item_metadata),
        'category': item_metadata['category'],
        'color': item_metadata['color'],
        'original_mask_bbox_xyxy': [x1, y1, x2, y2],
        'cutout_crop_bbox_xyxy': [cx1, cy1, cx2, cy2],
        'padding_px': CUTOUT_PADDING_PX,
        'original_bbox_width': x2 - x1,
        'original_bbox_height': y2 - y1,
        'aspect_ratio': round((x2 - x1) / max(1, y2 - y1), 4),
        'mask_area_ratio': round(float(mask.mean()), 4),
        'quality_valid': quality['valid'],
        'quality_reasons': ', '.join(quality['reasons']),
        'bbox_area_ratio': quality.get('bbox_area_ratio'),
        'proxy_coverage': quality.get('proxy_coverage'),
        'selection_score': quality.get('selection_score'),
    }

def save_outputs(rgb, mask, item_id, source_path, method, item_metadata, quality):
    record = build_output_record(
        mask,
        item_id,
        source_path,
        method,
        item_metadata,
        quality
    )
    if record is None:
        return None
    cx1, cy1, cx2, cy2 = record['cutout_crop_bbox_xyxy']
    rgba = np.dstack([
        rgb,
        (mask * 255).astype(np.uint8),
    ])
    Image.fromarray(
        (mask * 255).astype(np.uint8)
    ).save(MASK_DIR / f'{item_id}.png')
    Image.fromarray(
        rgba[cy1:cy2, cx1:cx2]
    ).save(CUTOUT_DIR / f'{item_id}.png')
    return record

## 5. Run Batch Segmentation and Save Transparent Cutouts

Process the product images, create product masks and transparent PNG cutouts, and save the successful results in a segmentation manifest. The code resumes from already completed images if the Colab runtime stops.

In [5]:
image_files = sorted(INPUT_DIR.glob('*.jpg'))

if MAX_IMAGES is not None:
    image_files = image_files[:MAX_IMAGES]

print(
    f'Processing {len(image_files):,} image files '
    f'(MAX_IMAGES={MAX_IMAGES}).'
)

manifest_path = OUTPUT_DIR / 'segmentation_manifest.csv'
failures_path = OUTPUT_DIR / 'segmentation_failures.csv'

if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    manifest['item_id'] = manifest['item_id'].astype(str)
    completed_ids = set(manifest['item_id'])
else:
    manifest = pd.DataFrame()
    completed_ids = set()

run_records = []
failures = []

for index, source_path in enumerate(tqdm(image_files, desc='Segmenting'), start=1):
    item_id = source_path.stem
    if item_id in completed_ids:
        continue
    try:
        rgb = np.asarray(
            Image.open(source_path).convert('RGB')
        )
        if min(rgb.shape[:2]) < 16:
            raise ValueError('image is too small')
        item_metadata = CATALOG_BY_ITEM_ID[item_id]
        mask, method, quality = choose_product_mask(
            rgb,
            item_metadata
        )
        record = save_outputs(
            rgb,
            mask,
            item_id,
            source_path,
            method,
            item_metadata,
            quality
        )
        if record is None:
            raise ValueError('empty mask bounding box')
        run_records.append(record)
        completed_ids.add(item_id)
    except Exception as exc:
        (MASK_DIR / f'{item_id}.png').unlink(missing_ok=True)
        (CUTOUT_DIR / f'{item_id}.png').unlink(missing_ok=True)
        failures.append({
            'item_id': item_id,
            'source_path': str(source_path),
            'error': str(exc),
        })
    # Save progress every 10 images.
    if index % 10 == 0:
        manifest = pd.concat(
            [manifest, pd.DataFrame(run_records)],
            ignore_index=True
        ).drop_duplicates('item_id', keep='last')
        manifest.to_csv(manifest_path, index=False)
        pd.DataFrame(
            failures,
            columns=['item_id', 'source_path', 'error']
        ).to_csv(failures_path, index=False)
        run_records = []
        torch.cuda.empty_cache()

# Save final progress.
manifest = pd.concat(
    [manifest, pd.DataFrame(run_records)],
    ignore_index=True
).drop_duplicates('item_id', keep='last')

if not manifest.empty:
    manifest = manifest.sort_values('item_id').reset_index(drop=True)

manifest.to_csv(manifest_path, index=False)

pd.DataFrame(
    failures,
    columns=['item_id', 'source_path', 'error']
).to_csv(failures_path, index=False)

print(
    f'Saved successful cutouts: {len(manifest):,}; '
    f'new quality/processing failures: {len(failures):,}.'
)

display(pd.DataFrame(failures).head(20))

Processing 3,762 image files (MAX_IMAGES=None).


Segmenting:   0%|          | 0/3762 [00:00<?, ?it/s]

Saved successful cutouts: 3,673; new quality/processing failures: 89.


,item_id,source_path,error
0,101864489,/content/drive/MyDrive/polyvore_image_cache/10...,quality_gate_failed: low_whole_object_coverage
1,103341484,/content/drive/MyDrive/polyvore_image_cache/10...,quality_gate_failed: low_whole_object_coverage
2,105518199,/content/drive/MyDrive/polyvore_image_cache/10...,quality_gate_failed: low_whole_object_coverage
3,105862435,/content/drive/MyDrive/polyvore_image_cache/10...,quality_gate_failed: low_whole_object_coverage
4,106278665,/content/drive/MyDrive/polyvore_image_cache/10...,quality_gate_failed: low_whole_object_coverage
5,106377661,/content/drive/MyDrive/polyvore_image_cache/10...,quality_gate_failed: low_whole_object_coverage
6,121598270,/content/drive/MyDrive/polyvore_image_cache/12...,quality_gate_failed: low_whole_object_coverage
7,132992976,/content/drive/MyDrive/polyvore_image_cache/13...,quality_gate_failed: low_whole_object_coverage
8,134116545,/content/drive/MyDrive/polyvore_image_cache/13...,quality_gate_failed: low_whole_object_coverage
9,136480573,/content/drive/MyDrive/polyvore_image_cache/13...,quality_gate_failed: low_whole_object_coverage


## 6. Create the 100-Image Quality Review Page

Create an HTML page showing a sample of transparent cutouts on a checkerboard background. The checkerboard represents transparency. A CSV scoring sheet is also created for manual evaluation.

In [6]:
REVIEW_SAMPLE_SIZE = 100

if manifest.empty:
    raise RuntimeError('No corrected cutouts available. Run the batch segmentation cell first.')

risk_routes = {'hybrid_required'}
risky = manifest[manifest['color_route'].isin(risk_routes)].copy()
risky = risky.sample(n=min(50, len(risky)), random_state=42)
remaining = manifest.drop(index=risky.index)
random_count = min(REVIEW_SAMPLE_SIZE - len(risky), len(remaining))

review = pd.concat(
    [risky, remaining.sample(n=random_count, random_state=42)],
    ignore_index=True
)

review = review.sample(frac=1, random_state=42).reset_index(drop=True)
review['score'] = ''
review.to_csv(REVIEW_DIR / 'review_scores.csv', index=False)
thumb_dir = REVIEW_DIR / 'thumbnails'
thumb_dir.mkdir(exist_ok=True)

cards = []

for index, row in review.iterrows():
    image = Image.open(row.cutout_path).convert('RGBA')
    draw = np.zeros((240, 240, 4), dtype=np.uint8)
    tile = 16
    for y in range(0, 240, tile):
        for x in range(0, 240, tile):
            shade = 210 if (x // tile + y // tile) % 2 else 245
            draw[y:y+tile, x:x+tile] = (shade, shade, shade, 255)
    canvas = Image.fromarray(draw, 'RGBA')
    fitted = ImageOps.contain(image, (220, 220))
    canvas.alpha_composite(
        fitted,
        ((240 - fitted.width) // 2, (240 - fitted.height) // 2)
    )
    thumb_name = f'{index:03d}.png'
    canvas.convert('RGB').save(thumb_dir / thumb_name)
    risk_label = (
        'HYBRID REVIEW'
        if row.color_route in risk_routes
        else 'representative sample'
    )
    cards.append(f"""<article><img src="thumbnails/{thumb_name}">
    <p><b>{index:03d}</b> - {html.escape(str(row.item_id))}</p>
    <p>Category: {html.escape(str(row.category))}; color: {html.escape(str(row.color))}</p>
    <p>Method: {html.escape(str(row.method))}; route: {html.escape(str(row.color_route))}</p>
    <p><b>{risk_label}</b></p>
    <p>Score: ☐ 0 failed &nbsp; ☐ 1 rough &nbsp; ☐ 2 clean</p></article>""")

page = """<!doctype html><html><head><meta charset="utf-8">
<title>Week 3 Corrected Mask Quality Review</title>
<style>body{font-family:Arial;margin:24px;background:#fafafa}.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(245px,1fr));gap:14px}article{background:white;padding:10px;border:1px solid #ddd;border-radius:8px}img{width:100%;height:240px;object-fit:contain}p{font-size:12px;margin:6px 0}</style>
</head><body><h1>Week 3 Corrected Mask Quality Review</h1>
<p>Checkerboard areas are transparent. Score 0 = failed, 1 = usable but rough, 2 = clean full product. Target: at least 80% score 1 or 2.</p>
<div class="grid">""" + ''.join(cards) + '</div></body></html>'

(REVIEW_DIR / 'index.html').write_text(page, encoding='utf-8')

print(f'Review page: {REVIEW_DIR / "index.html"}')
print(f'Scoring sheet: {REVIEW_DIR / "review_scores.csv"}')

Review page: /content/drive/MyDrive/polyvore_segmented_cache/review/index.html
Scoring sheet: /content/drive/MyDrive/polyvore_segmented_cache/review/review_scores.csv


## 7. Calculate the Final Quality Evaluation Results

Read the 100 manual review scores and calculate the usable-cutout rate and clean for collage rate. The MVP target is at least 80% usable cutouts.

In [5]:
import pandas as pd
from pathlib import Path
REVIEW_DIR = Path('/content/drive/MyDrive/polyvore_segmented_cache/review')
scores = pd.read_csv(REVIEW_DIR / 'review_scores.csv')
scored = pd.to_numeric(scores['score'], errors='coerce').dropna()
assert scored.isin([0, 1, 2]).all(), 'Scores must be 0, 1, or 2.'
if scored.empty:
    print('No scores entered yet.')
else:
    usable_rate = (scored >= 1).mean()
    clean_rate = (scored == 2).mean()
    print(f'Reviewed: {len(scored)}/{len(scores)}')
    print(f'Usable cutouts: {usable_rate:.1%} (target: at least 80.0%)')
    print(f'Clean for collage: {clean_rate:.1%}')
    display(scores['score'].value_counts(dropna=False).sort_index())

Reviewed: 100/100
Usable cutouts: 100.0% (target: at least 80.0%)
Clean for collage: 96.0%


,count
score,
1,4
2,96
